# AntMaze (medium maze, play data): BC, IQL and online fine-tuning

Long-horizon navigation with a quadruped: an 8-joint ant has to reach a distant goal in a maze with a sparse reward. Dataset: Minari `D4RL/antmaze/medium-play-v2`.

## Problem formulation

| | |
|---|---|
| **State** $s \in \mathbb{R}^{31}$ | position $(x, y)$; 27 body features (torso height, orientation quaternion, 8 joint angles, 6 torso velocities, 8 joint velocities); goal $(g_x, g_y)$ |
| **Action** $a \in [-1, 1]^{8}$ | hip and ankle torques of the four legs |
| **Reward** | $r = 0$ within 0.45 of the test goal, $r = -1$ otherwise (relabeled) |
| **Termination** | goal reached ($d = 1$) |
| **Discount** | $\gamma = 0.995$ |

## Dataset
About $10^6$ transitions of the ant walking between hand-selected locations ("play" data). No trajectory connects the evaluation start to the evaluation goal, so a successful policy has to combine segments of different trajectories.

**Reward relabeling.** As in D4RL, every transition is relabeled for the evaluation goal (`relabel_to_test_goal`) and the goal in the state is replaced accordingly. The evaluation goal is fixed at the center of the goal cell (`FixedGoal`).

## Evaluation protocol
Minari's evaluation maze fixes the start cell (`r`) and the goal cell (`g`); the start position receives uniform noise of ±1 (a quarter cell). Episodes last at most 1,000 steps. For some Minari AntMaze datasets the stored evaluation maze is rotated relative to the data; `make_eval_env` selects the orientation consistent with the dataset positions.

**Metrics.** Success rate from the standard start, the fraction of episodes in which the ant ends upside down (`flipped`), and the success rate from every free start cell. During fine-tuning, exploration episodes are reset when the ant flips (`reset_on_flip`); the reset is stored as a truncation, not a termination.

## Offline RL setting
Offline reinforcement learning: the policy is learned from a fixed dataset $\mathcal{D} = \{(s, a, r, s', d)\}$ collected by an unknown behavior policy, without interaction with the environment. The environment is used for evaluation only. Value estimates are reliable only for actions represented in $\mathcal{D}$; the methods below avoid querying the critic on out-of-distribution actions.

## Usage
1. Copy the project folder (`offline-rl-jax/`) to Google Drive, e.g. `MyDrive/offline-rl-jax`.
2. Open this notebook in Colab with a GPU runtime (`Runtime → Change runtime type → T4 GPU`).
3. Set `PROJECT_DIR` in Section 1 if the folder is elsewhere, then run all cells.

Outputs are written to the project folder: `results/<name>/` (JSON and figures, tracked by git) and `checkpoints/<name>/` (network weights, not tracked).

`RUN_OFFLINE` (Section 2): `True` runs Part 1 and Part 2; `False` loads `checkpoints/<name>/offline.pkl` and runs Part 2 only.

## 1. Setup
Mounts Google Drive, sets the working directory and installs the simulator and dataset packages (JAX is preinstalled on Colab).

In [ ]:
import os, sys, shutil
PROJECT_DIR = "/content/drive/MyDrive/offline-rl-jax"      # the project folder you uploaded to Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:                                        # running locally instead of Colab
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
if shutil.which("nvidia-smi"):
    os.environ["MUJOCO_GL"] = "egl"                        # headless rendering (only needed for videos)
print("project folder:", PROJECT_DIR)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import numpy as np
import jax
import matplotlib.pyplot as plt
for _m in [m for m in sys.modules if m == "offrl" or m.startswith("offrl.")]:
    del sys.modules[_m]           # always use the code currently in PROJECT_DIR (not a copy cached by an earlier run)
import offrl
from offrl import (Task, Normalizer, to_device, train_offline, EnvPool, evaluate, evaluate_final, summarize,
                   load_config, save_checkpoint, load_checkpoint, save_json, load_json, select_seed)
from offrl.algos.finetune import finetune
from offrl import plotting
print("offrl", offrl.__version__, "from", os.path.dirname(offrl.__file__))
print("JAX", jax.__version__, "| devices:", jax.devices())

## 2. Settings
Hyperparameters are read from `configs/antmaze_medium_play.yaml`. With `RUN_OFFLINE = False`, `cfg` is taken from the checkpoint so that it matches the saved networks; `ONLINE` is always read from the config file.

In [ ]:
CONFIG_FILE = "configs/antmaze_medium_play.yaml"
RUN_OFFLINE = True     # True: train offline (Part 1), save, then fine-tune. False: load the saved offline networks.

cfg, ONLINE = load_config(CONFIG_FILE)
RESULTS_DIR = f"results/{cfg['name']}"            # small files (JSON, figures), meant for GitHub
CKPT_DIR = f"checkpoints/{cfg['name']}"           # trained networks (large, ignored by git)
if not RUN_OFFLINE:
    ck = load_checkpoint(f"{CKPT_DIR}/offline.pkl")
    cfg = ck["config"]
print("offline:", cfg)
print("online: ", ONLINE)

## 3. Data and evaluation environment
`Task` defines the dataset, the evaluation environment, the state representation and the transition labels. States are standardized with the dataset statistics (or those saved with the checkpoint), and the dataset is stored on the GPU. `evaluate_fn` is the evaluation used during training (fixed environment seeds, mean action).

In [ ]:
task = Task(cfg)
raw = task.transitions()
normalizer = Normalizer(raw["states"]) if RUN_OFFLINE else Normalizer.from_state_dict(ck["normalizer"])
DATA = to_device(raw, normalizer)
pool = EnvPool(task)
N_SEEDS = len(cfg["seeds"])
print("state dim:", DATA["s"].shape[1], "| action dim:", DATA["a"].shape[1])

def evaluate_fn(pi):      # the test used during training: deterministic policy mean, eval_episodes per seed
    return evaluate(task, pi, N_SEEDS, cfg["eval_episodes"], 1000, pool, normalizer)

Maze layout with 30 dataset trajectories and the start and goal of one evaluation episode.

In [ ]:
from offrl.envs import draw_maze
obs0, _ = task.env.reset(seed=0)
fig, ax = plt.subplots(figsize=(5, 5))
draw_maze(ax, task.geom)
for e in task.dataset.sample_episodes(min(30, task.dataset.total_episodes)):
    xy = e.observations["achieved_goal"]; ax.plot(xy[:, 0], xy[:, 1], lw=0.8, alpha=0.7)
ax.plot(*obs0["achieved_goal"], "ko", ms=8, label="test start" if task.is_ant else "a test start (random each episode)")
ax.plot(*obs0["desired_goal"], "*", color="gold", ms=20, mec="k", label="test goal")
ax.legend(loc="upper right"); ax.set_title("30 dataset trajectories"); plt.show()

# Part 1: Offline training

## 4. Methods

Both methods train a policy $\pi_\theta(a \mid s)$ on $\mathcal{D}$. All networks are MLPs with two hidden layers of `hidden_dim` units and ReLU activations; the policy mean is passed through $\tanh$ to the action range $[-1, 1]$.

### Behavior cloning (BC)
Regression of the dataset action (`offrl/algos/bc.py`):
$$\mathcal{L}_{BC}(\theta) = \mathbb{E}_{(s,a)\sim\mathcal{D}}\big[\,\lVert \mu_\theta(s) - a \rVert^2\,\big]$$
BC imitates the behavior policy and does not use rewards.

### Implicit Q-Learning (IQL)
Kostrikov, Nair and Levine, *Offline Reinforcement Learning with Implicit Q-Learning*, ICLR 2022 ([arXiv:2110.06169](https://arxiv.org/abs/2110.06169)); `offrl/algos/iql.py`. Three networks are updated on each batch.

**State value** by expectile regression on the target Q-values of dataset actions:
$$\mathcal{L}_V(\psi) = \mathbb{E}_{(s,a)\sim\mathcal{D}}\Big[\,L_2^{\tau}\big(\hat Q(s,a) - V_\psi(s)\big)\Big], \qquad L_2^{\tau}(u) = \lvert \tau - \mathbf{1}(u<0)\rvert\, u^2, \qquad \hat Q = \min(Q_{\bar\phi_1}, Q_{\bar\phi_2})$$
For $\tau = 0.5$ this is mean regression; for $\tau > 0.5$ (`expectile`), $V(s)$ approaches the upper range of the Q-values of in-distribution actions, an in-sample estimate of $\max_a Q(s,a)$.

**Action value** by temporal-difference learning with $V(s')$ in place of $\max_{a'} Q(s',a')$:
$$\mathcal{L}_Q(\phi_i) = \mathbb{E}_{(s,a,r,s',d)\sim\mathcal{D}}\Big[\big(r + \gamma\,(1-d)\,V_\psi(s') - Q_{\phi_i}(s,a)\big)^2\Big], \quad i = 1, 2$$
Two critics are used; their target networks are updated as $\bar\phi \leftarrow (1-\tau_t)\bar\phi + \tau_t\phi$ with $\tau_t$ = `tau`. Terminal flags $d = 1$ mark true terminations only, not time limits.

**Policy** by advantage-weighted regression:
$$\mathcal{L}_\pi(\theta) = -\,\mathbb{E}_{(s,a)\sim\mathcal{D}}\Big[\min\big(e^{\beta\,(\hat Q(s,a) - V_\psi(s))},\ 100\big)\,\log \pi_\theta(a \mid s)\Big]$$
Dataset actions are weighted by their exponentiated advantage $A = \hat Q - V$; $\beta$ (`beta`) controls the selectivity ($\beta \to 0$ recovers BC). The policy is Gaussian, $\mathcal{N}(\mu_\theta(s), \sigma^2)$, with a learned state-independent $\sigma$; evaluation uses the mean action $\mu_\theta(s)$. The policy learning rate follows a cosine decay over `iql_steps`.

Hyperparameters (config file): `expectile` ($\tau$), `beta` ($\beta$), `discount` ($\gamma$), `tau` (target update rate), `lr`, `batch_size`, `hidden_dim`.

### Training
Seeds are trained in parallel (`jax.vmap`); each block of `eval_every` updates is compiled as one program (`jax.lax.scan`). `train` in the log is training time excluding evaluation; the first block includes compilation. For IQL the log reports the TD loss `q_loss` and the batch mean of $V(s)$.

### 4a. Behavior cloning

In [ ]:
if RUN_OFFLINE:
    bc_states, bc_log = train_offline("bc", cfg, DATA, evaluate_fn)

### 4b. Implicit Q-Learning

In [ ]:
if RUN_OFFLINE:
    iql_states, iql_log = train_offline("iql", cfg, DATA, evaluate_fn)

## 5. Offline results
Final evaluation: `final_episodes` episodes per seed with the mean action, and a success map from every free start cell (`episodes_per_cell` episodes each). Saved to `results/<name>/`.
- **Training curves and final test.** Mean over seeds with a ±1 std band; bars show mean ± std over seeds.
- **Episode outcomes.** Outcome of every final-test episode and the number of steps to reach the goal.
- **Success maps.** Success rate by start cell, averaged over seeds.
- **Value function.** $V(s)$ of IQL (seed 0) over the maze for the test goal.
- **Rollouts.** 20 test episodes of seed 0 (green: goal reached; red: failed, ending in × if flipped, ■ at the time limit) and the distance to the goal over time.

In [ ]:
if RUN_OFFLINE:
    offline_res = evaluate_final(task, cfg, {"BC": bc_states, "IQL": iql_states}, {"BC": bc_log, "IQL": iql_log},
                                 pool, normalizer)
    summarize(offline_res)
    save_json(f"{RESULTS_DIR}/offline_results.json", offline_res)
    plotting.plot_offline_results(offline_res, save_path=f"{RESULTS_DIR}/figures/offline.png")
    plotting.plot_episode_outcomes(offline_res, save_path=f"{RESULTS_DIR}/figures/offline_episodes.png")

In [ ]:
if RUN_OFFLINE:
    plotting.plot_success_maps(plotting.maps_from_results(offline_res), task.geom, offline_res["goal"],
                               save_path=f"{RESULTS_DIR}/figures/offline_success_maps.png")
    plotting.plot_value_heatmap(task, select_seed(iql_states["v"], 0), normalizer,
                                save_path=f"{RESULTS_DIR}/figures/iql_value.png")
    plotting.plot_rollouts(task, {"BC": bc_states, "IQL": iql_states}, pool, normalizer, n=20,
                           save_path=f"{RESULTS_DIR}/figures/offline_rollouts.png")

In [ ]:
if RUN_OFFLINE:
    from offrl.video import plot_filmstrip
    try:
        for name, st in {"BC": bc_states, "IQL": iql_states}.items():
            plot_filmstrip(task, st, normalizer, title=name, save_path=f"{RESULTS_DIR}/figures/offline_filmstrip_{name.replace(' ', '_')}.png")
    except Exception as e:                      # rendering needs a GPU runtime on Colab (MUJOCO_GL=egl)
        print("Rendering is not available here:", e)

## 6. Checkpoint
Saves the configuration, the state normalizer and the BC and IQL networks (including optimizer states) to `checkpoints/<name>/offline.pkl`.

In [ ]:
if RUN_OFFLINE:
    save_checkpoint(f"{CKPT_DIR}/offline.pkl", config=cfg, online=ONLINE, normalizer=normalizer.state_dict(),
                    bc_states=bc_states, iql_states=iql_states)

# Part 2: Online fine-tuning

The offline IQL networks are trained further with environment interaction, using the same losses (`offrl/algos/finetune.py`, following the official IQL implementation):

- **Data collection.** `envs_per_seed` environments per seed; actions $a = \operatorname{clip}\big(\mu_\theta(s) + T\,\sigma\,\varepsilon,\ -1,\ 1\big)$, $\varepsilon \sim \mathcal{N}(0, I)$, with $T$ = `explore_temperature`.
- **Labels.** Online transitions are labeled exactly as the offline data.
- **Replay.** One buffer per seed containing $\mathcal{D}_{\text{offline}} \cup \mathcal{D}_{\text{online}}$, sampled uniformly.
- **Optimization.** One update per environment step (`updates_per_step`); optimizers are re-initialized with a constant learning rate.

The log reports the evaluation metrics every `eval_every` online steps (step 0 is the offline policy) and statistics of the exploration episodes (`explore`). Checkpoints are written to `checkpoints/<name>/finetuned.pkl` after each evaluation.

## 7. Initialization from the offline IQL networks

In [ ]:
if not RUN_OFFLINE:
    iql_states = ck["iql_states"]
offline_iql = iql_states
print("starting fine-tuning from:", "memory (Part 1)" if RUN_OFFLINE else f"{CKPT_DIR}/offline.pkl")
print("online settings:", ONLINE)

## 8. Fine-tuning
Evaluation metrics during fine-tuning (mean over seeds, ±1 std band); step 0 is the offline policy.

In [ ]:
def save_progress(states, log):          # called after every evaluation
    save_checkpoint(f"{CKPT_DIR}/finetuned.pkl", config=cfg, online=ONLINE, normalizer=normalizer.state_dict(),
                    iql_states=states, log=log)

ft_states, ft_log = finetune(task, cfg, ONLINE, offline_iql, DATA, pool, normalizer, on_eval=save_progress)
save_json(f"{RESULTS_DIR}/finetune_log.json", ft_log)
plotting.plot_finetune(ft_log, list(task.curve_metrics), save_path=f"{RESULTS_DIR}/figures/finetune_curves.png")

### Offline and online learning curve
IQL evaluation metric over offline training followed by fine-tuning (shaded; one environment step corresponds to one gradient step). Reference lines: final BC result. The offline curve is read from `results/<name>/offline_results.json`.

In [ ]:
off_res = offline_res if RUN_OFFLINE else load_json(f"{RESULTS_DIR}/offline_results.json")
bc_final = float(np.mean([r["final"][task.primary] for r in off_res["methods"]["BC"]]))
refs = {task.primary: {"BC (final test)": bc_final}}
if task.kind == "mujoco":
    refs[task.primary]["dataset mean"] = task.data_return
plotting.plot_offline_online(off_res, ft_log, list(task.curve_metrics), refs=refs,
                             save_path=f"{RESULTS_DIR}/figures/offline_to_online.png")

## 9. Offline and fine-tuned policies
Final evaluation of both policies on the same episodes, with the figures of Section 5.

In [ ]:
ft_res = evaluate_final(task, {**cfg, "final_episodes": ONLINE["final_episodes"]},
                        {"offline IQL": offline_iql, "fine-tuned IQL": ft_states}, None, pool, normalizer)
summarize(ft_res)
save_json(f"{RESULTS_DIR}/finetune_results.json", ft_res)
plotting.plot_offline_results(ft_res, save_path=f"{RESULTS_DIR}/figures/finetune_final.png")
plotting.plot_episode_outcomes(ft_res, save_path=f"{RESULTS_DIR}/figures/finetune_episodes.png")

In [ ]:
plotting.plot_success_maps(plotting.maps_from_results(ft_res), task.geom, ft_res["goal"],
                           save_path=f"{RESULTS_DIR}/figures/finetune_success_maps.png")
plotting.plot_rollouts(task, {"offline IQL": offline_iql, "fine-tuned IQL": ft_states}, pool, normalizer, n=20,
                       save_path=f"{RESULTS_DIR}/figures/finetune_rollouts.png")

In [ ]:
from offrl.video import plot_filmstrip
try:
    for name, st in {"offline IQL": offline_iql, "fine-tuned IQL": ft_states}.items():
        plot_filmstrip(task, st, normalizer, title=name, save_path=f"{RESULTS_DIR}/figures/finetune_filmstrip_{name.replace(' ', '_')}.png")
except Exception as e:                      # rendering needs a GPU runtime on Colab (MUJOCO_GL=egl)
    print("Rendering is not available here:", e)

## 10. Episode recording (optional)
Records one evaluation episode as a GIF (requires rendering). Top-down camera by default; `top_down=False` follows the ant. `seed_index` selects the trained seed.

In [ ]:
from offrl.video import record_gif
from IPython.display import Image, display
# gif_path, _ = record_gif(task, ft_states, normalizer, f"{RESULTS_DIR}/figures/fine_tuned.gif", seed_index=0)
# display(Image(filename=gif_path))